# Replica della PCA varimax di Thiebaut de Schotten et al. 2020 sul disconnettoma voxelwise

**Domanda.** Se si applica al disconnettoma voxelwise `s2.4-vol` lo stesso metodo del paper (PCA sulla covarianza, poi rotazione varimax), quante componenti servono per spiegare la varianza, e quanto ogni componente dipende dalla dimensione della lesione?

**Cosa legge.** La matrice `matrix.npy` e `metadata.csv` della sessione `s2.4-vol` (5845 soggetti × 291.345 voxel) e la colonna `lesion_volume_voxels_2mm` di `assets/metadata/participants.csv`.

**Cosa scrive.** Solo una cache in `results/pipeline_building/thiebaut_2020_replication/<sessione>/` (cartella ignorata da git), per non rifare ogni volta i circa 6 minuti di calcolo (PCA e varimax) e i circa 3 delle mappe. Il calcolo vero sta in `src/analysis/covariance_pca.py`, con i suoi test.

**Cosa non fa.** Non cambia nessuna matrice. Non usa atlanti né regioni: lavora sui voxel. Non confronta la lesione con il disconnettoma.

> **Rispetto al paper.** Il notebook non cerca di riprodurre i numeri del paper: il paper usa 372 regioni, qui i voxel sono 291.345, quindi si confrontano le tendenze e si riporta quello che esce.

| Sezione | Contenuto |
|---|---|
| 1 | Scelta della matrice e dei parametri |
| 2 | PCA sulla covarianza |
| 3 | Quante componenti servono per il 50%, l'80%, il 90% della varianza |
| 4 | Rotazione varimax |
| 5 | Quanto ogni componente dipende dal volume della lesione |
| 6 | Mappe delle componenti nel cervello e loro riproducibilità |

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from nilearn import plotting

# La radice del repo è la prima cartella, risalendo da quella corrente, che contiene assets/metadata. Non dipende da
# quante volte si esegue la cella: l'os.chdir qui sotto sposta la cartella corrente nella radice.
_start = Path.cwd().resolve()
NOTEBOOK_REPO_ROOT = next((p for p in [_start, *_start.parents] if (p / "assets" / "metadata").is_dir()), None)
if NOTEBOOK_REPO_ROOT is None:
    raise FileNotFoundError(f"nessuna cartella con assets/metadata sopra {_start}: il notebook va aperto dentro il repo")
if str(NOTEBOOK_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_REPO_ROOT))
# load_participants_registry legge assets/metadata/participants.csv con un path relativo alla repo root.
os.chdir(NOTEBOOK_REPO_ROOT)

from src.analysis.covariance_pca import (
    CovariancePca,
    component_t_maps,
    fit_covariance_pca,
    map_reproducibility,
    positive_peak_signs,
    squared_correlation,
    varimax_scores,
)
from src.utils.participants import load_participants_registry

# src.analysis.* imposta matplotlib.use("Agg") al proprio import: senza questo i plot non compaiono.
%matplotlib inline

sns.set_theme(style="whitegrid")

## 1. Scelta della matrice e dei parametri

Questi sono i valori da modificare:

- `SESSION_DIR`: la cartella della matrice. Qui `s2.4-vol`, il disconnettoma voxelwise dei 5845 soggetti ammessi.

- `VARIANCE_SHARE_FOR_VARIMAX`: la quota di varianza da raggiungere per decidere quante componenti ruotare. Con `0.9` si ruotano le prime componenti che insieme spiegano almeno il 90%.

  > Nel paper (Risultati): *"30 components, out of 46 in total, explained more than 90% of the variance of the regions disconnected by a stroke (Fig. 2a)"*. Il paper non dichiara come ha scelto il numero di componenti: lo legge dalla curva della varianza cumulata. Qui si usa la stessa soglia come criterio.

- `VARIMAX_MAX_ITER`: iterazioni massime della rotazione. Qui 500.

  > Nel paper (Metodi, «Data compression»): *"a PCA using a covariance matrix and varimax rotation (with a maximum of 500 iterations for convergence)"*.

- `LESION_VOLUME_COLUMN`: la colonna del volume della lesione, in voxel a 2 mm.

- `USE_CACHE`: con `True` riusa il calcolo già fatto, con `False` lo rifà e sovrascrive la cache.

In [ ]:
SESSION_DIR = Path("data/derived/sdc_matrix/07-10_s2.4-vol")
VARIANCE_SHARE_FOR_VARIMAX = 0.9
VARIMAX_MAX_ITER = 500
LESION_VOLUME_COLUMN = "lesion_volume_voxels_2mm"
USE_CACHE = True

CACHE_DIR = Path("results/pipeline_building/thiebaut_2020_replication") / SESSION_DIR.name

# La matrice resta su disco (mmap): ogni calcolo la legge a blocchi di colonne, mai tutta insieme.
X = np.load(SESSION_DIR / "matrix.npy", mmap_mode="r")
metadata = pd.read_csv(SESSION_DIR / "metadata.csv")
if len(metadata) != X.shape[0]:
    raise ValueError(f"metadata.csv ha {len(metadata)} righe ma la matrice ha {X.shape[0]} soggetti")

print(f"Sessione: {SESSION_DIR.name}")
print(f"Matrice: {X.shape[0]} soggetti x {X.shape[1]} voxel, tipo {X.dtype}")
print(f"Soggetti per dataset:")
for name, count in metadata["dataset"].value_counts().sort_index().items():
    print(f"  {name}: {count}")

## 2. PCA sulla covarianza

**Cosa fa la PCA.** Trova le direzioni lungo cui i soggetti differiscono di più. La prima componente è la direzione con più variabilità, la seconda è la più variabile tra quelle indipendenti dalla prima, e così via. Ogni componente ha una *varianza*: più è grande, più quella direzione distingue i soggetti.

**Covarianza, senza standardizzare.** Ogni voxel viene centrato (si toglie la sua media sui soggetti) ma non diviso per la sua deviazione standard. Un voxel che varia molto tra i soggetti pesa quindi di più di uno che varia poco.

**Perché la covarianza.** Ogni voxel è una probabilità di disconnessione, tutti sulla stessa scala: che un voxel vari più di un altro è informazione vera, non un accidente di unità di misura.

**Se standardizzassimo** (PCA sulla *correlazione*). Ogni voxel peserebbe uguale, anche quelli quasi sempre a zero, la cui variabilità minuscola avrebbe lo stesso peso delle altre. Le prime componenti smetterebbero di seguire dove la disconnessione cambia davvero tra i soggetti.

> Nel paper (Metodi, «Data compression»): *"Each matrix entered a PCA using a covariance matrix and varimax rotation (with a maximum of 500 iterations for convergence) in order to estimate the number of principal components to extract for the four conditions."* Il testo specifica solo «covariance matrix»: non dice altro sulla standardizzazione.

**Perché non è la PCA di sempre.** Il calcolo diretto costruirebbe una tabella che confronta i voxel tra loro: 291.345 × 291.345 valori, circa 680 GB. Non entra in memoria.

Ma i soggetti sono solo 5845. Si può costruire invece una tabella che confronta i **soggetti** tra loro (5845 × 5845, circa 270 MB). Le due tabelle hanno **gli stessi autovalori**, cioè la stessa varianza per componente, e dalla piccola si ricavano i pesi dei voxel.

Il risultato è la stessa PCA, calcolata per la strada più corta. I test lo confermano confrontandola con `sklearn.decomposition.PCA` su matrici piccole.

Il calcolo dura circa 4 minuti la prima volta.

In [ ]:
PCA_CACHE = CACHE_DIR / "pca.npz"
FINGERPRINT = CACHE_DIR / "source.json"

# La cache sa da quale matrice viene: se la matrice cambia, la cache non vale più e il notebook si ferma.
source = {
    "session": SESSION_DIR.name,
    "shape": list(X.shape),
    "matrix_mtime_ns": (SESSION_DIR / "matrix.npy").stat().st_mtime_ns,
}
if USE_CACHE and PCA_CACHE.is_file():
    if json.loads(FINGERPRINT.read_text()) != source:
        raise ValueError(f"la cache in {CACHE_DIR} viene da un'altra matrice: cancella la cartella o metti USE_CACHE = False")
    cached = np.load(PCA_CACHE)
    pca = CovariancePca(cached["eigenvalues"], cached["left_vectors"], cached["column_means"])
    print(f"PCA letta dalla cache: {PCA_CACHE}")
else:
    pca = fit_covariance_pca(X)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(PCA_CACHE, eigenvalues=pca.eigenvalues, left_vectors=pca.left_vectors, column_means=pca.column_means)
    FINGERPRINT.write_text(json.dumps(source))
    print(f"PCA calcolata e salvata in {PCA_CACHE}")

print(f"Componenti con varianza non nulla: {pca.n_nonzero_components()}")
print(f"Varianza totale della matrice: {pca.eigenvalues.sum():.1f}")

## 3. Quante componenti servono per il 50%, l'80%, il 90% della varianza

La **varianza cumulata** di *k* componenti è la somma delle quote di varianza delle prime *k*. Esempio: se la prima componente spiega il 26% e la seconda il 25%, insieme ne spiegano il 51%, quindi con 2 componenti si supera il 50%.

La tabella dice, per ogni soglia, il numero minimo di componenti che la raggiunge.

> Nel paper (Risultati): *"we modelled the main profiles of brain disconnections and revealed that 30 components, out of 46 in total, explained more than 90% of the variance of the regions disconnected by a stroke (Fig. 2a)"*. È la stessa lettura: 30 componenti su 46 per il 90%.
>
> E nei Metodi: *"Plot of the components according to their cumulative explained variance can be seen in Fig. 2a."*

In [ ]:
THRESHOLDS = (0.5, 0.8, 0.9)
n_for_share = {share: pca.n_components_for(share) for share in THRESHOLDS}

for share, n_components in n_for_share.items():
    print(f"Per spiegare almeno il {share:.0%} della varianza servono {n_components} componenti su {pca.n_nonzero_components()}")

print()
ratio = pca.explained_variance_ratio
cumulative = pca.cumulative_variance_ratio
for k in range(10):
    print(f"Componente {k + 1}: {ratio[k]:.1%} della varianza, {cumulative[k]:.1%} con le componenti fino a questa")

In [ ]:
N_SHOWN = 150
x = np.arange(1, N_SHOWN + 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, cumulative[:N_SHOWN], color="tab:blue")
for share, n_components in n_for_share.items():
    ax.axhline(share, color="grey", linestyle=":", linewidth=1)
    ax.plot([n_components], [cumulative[n_components - 1]], "o", color="tab:red")
    ax.annotate(f"{share:.0%}: {n_components} comp.", (n_components, cumulative[n_components - 1]),
                textcoords="offset points", xytext=(8, -14))
ax.set_xlabel("Numero di componenti")
ax.set_ylabel("Varianza cumulata")
ax.set_title(f"{SESSION_DIR.name}: varianza cumulata (prime {N_SHOWN} componenti)")
ax.set_ylim(0, 1.02)
plt.show()

## 4. Rotazione varimax

**Il problema.** La PCA dà componenti ordinate per varianza, ma ognuna è un mix di tutto il cervello: pesa su quasi tutti i voxel, con segni diversi. Dire cosa rappresenta è difficile.

**L'idea.** Le componenti sono assi, e gli assi si possono ruotare: i soggetti restano dove sono, cambia solo il riferimento con cui li descriviamo. La varianza totale delle componenti non cambia, cambia come è divisa tra loro.

**Cosa cerca il varimax.** Tra tutte le rotazioni, quella in cui ogni voxel pesa soprattutto su una componente e quasi zero sulle altre. Ogni componente diventa un gruppo di voxel che si disconnettono insieme: una zona.

**Se non ruotassimo.** Le componenti resterebbero valide e ordinate, ma ognuna sarebbe un mix di tutto il cervello, difficile da nominare. Non si perderebbe informazione, solo leggibilità.

**Il punteggio.** Ogni soggetto ha un valore per voxel, e ogni componente ruotata ha un peso per voxel. Il **punteggio** di un soggetto su una componente è un numero che dice quanto la riga del soggetto segue i pesi di quella componente. Con 42 componenti ogni soggetto ha 42 punteggi, al posto di 291.345 valori.

Si calcolano cercando i 42 numeri che, moltiplicati per le componenti e sommati, ricostruiscono al meglio la riga del soggetto. È una regressione, detta multipla perché tutte le componenti entrano insieme: dopo il varimax si sovrappongono un po', e calcolarle una alla volta conterebbe due volte la parte condivisa.

**A cosa servono.** Ogni componente, da sola, dice solo quali voxel pesa. Per collegarla a qualcosa che si misura sul soggetto serve sapere quanto ne ha ciascun soggetto, e questo sono i punteggi. Nella sezione 5 si confrontano con il volume della lesione, nella sezione 6 danno le mappe nel cervello.

Il numero di componenti da ruotare viene dalla soglia scelta all'inizio (`VARIANCE_SHARE_FOR_VARIMAX`).

> **Nel paper.** Il varimax compare nei Risultati: *"Using a varimax-rotated principal component analysis (PCA) we modelled the main profiles of brain disconnections"*. Il testo principale non spiega perché lo usa.
>
> I punteggi per regressione multipla, nei Metodi («Data compression»): *"Component scores were systematically extracted for all components identified by the PCA by means of multiple regression."*

In [ ]:
N_COMPONENTS = pca.n_components_for(VARIANCE_SHARE_FOR_VARIMAX)
print(f"Ruoto le prime {N_COMPONENTS} componenti (quelle che insieme spiegano almeno il {VARIANCE_SHARE_FOR_VARIMAX:.0%})")

VARIMAX_CACHE = CACHE_DIR / "varimax.npz"
varimax_source = {**source, "n_components": N_COMPONENTS, "max_iter": VARIMAX_MAX_ITER}
VARIMAX_FINGERPRINT = CACHE_DIR / "varimax_source.json"

if USE_CACHE and VARIMAX_CACHE.is_file():
    if json.loads(VARIMAX_FINGERPRINT.read_text()) != varimax_source:
        raise ValueError(f"la cache del varimax in {CACHE_DIR} ha altri parametri: metti USE_CACHE = False per rifarla")
    cached = np.load(VARIMAX_CACHE)
    scores, rotated_loadings = cached["scores"], cached["rotated_loadings"]
    print(f"Varimax letto dalla cache: {VARIMAX_CACHE}")
else:
    scores, rotated_loadings = varimax_scores(X, pca, n_components=N_COMPONENTS, max_iter=VARIMAX_MAX_ITER)
    np.savez(VARIMAX_CACHE, scores=scores, rotated_loadings=rotated_loadings)
    VARIMAX_FINGERPRINT.write_text(json.dumps(varimax_source))
    print(f"Varimax calcolato e salvato in {VARIMAX_CACHE}")

print(f"Punteggi: {scores.shape[0]} soggetti x {scores.shape[1]} componenti")
print(f"Pesi ruotati: {rotated_loadings.shape[0]} voxel x {rotated_loadings.shape[1]} componenti")

**Il prezzo: l'ordine.** Dopo la rotazione la prima componente non è più la più importante. I punteggi hanno tutti varianza 1, quindi non servono per ordinarle. Si guarda invece quanta varianza dei dati spiega ogni profilo: la somma dei pesi al quadrato della sua colonna, divisa per la varianza totale. La somma sulle componenti è la stessa di prima della rotazione; cambia come è ripartita.

**Il secondo prezzo: i gruppi potrebbero non esistere.** Il varimax sceglie la descrizione più semplice anche quando nei dati non ci sono gruppi. Che quelli trovati siano reali lo dice la riproducibilità tra metà dei soggetti (sezione 6).

**Come si legge la tabella.** Ogni riga è un posto nella graduatoria per varianza: a sinistra la quota della componente PCA di quel posto, a destra la componente ruotata che lo occupa, con la sua quota.

Il numero (*n.*) è l'etichetta della componente ruotata, quella usata nelle sezioni seguenti: non è la componente PCA con lo stesso numero.

In [ ]:
# Quota di varianza spiegata da ogni componente ruotata: somma dei pesi al quadrato della sua colonna,
# divisa per la varianza totale. Prima della rotazione la stessa formula dà la quota della PCA.
rotated_share = (rotated_loadings**2).sum(axis=0) / pca.eigenvalues.sum()
by_variance = np.argsort(rotated_share)[::-1]

print(f"Varianza spiegata dalle {N_COMPONENTS} componenti: {rotated_share.sum():.1%} dopo la rotazione, "
      f"{pca.cumulative_variance_ratio[N_COMPONENTS - 1]:.1%} prima (il totale non cambia, cambia come è ripartita)")
print()
print(f"{'Posto':>6} | {'PCA':>8} | Componente ruotata")
for place, index in enumerate(by_variance[:10], start=1):
    print(f"{place:>5}º | {pca.explained_variance_ratio[place - 1]:>8.1%} | n. {index + 1:>2}, {rotated_share[index]:.1%}")

## 5. Quanto ogni componente dipende dal volume della lesione

**Cos'è R².** Per ogni componente si guarda quanto il suo punteggio cresce o cala insieme al volume della lesione. R² è il quadrato della correlazione, e si legge come la quota della variabilità del punteggio che si spiega conoscendo il volume.

- R² = 0.04: il volume spiega il 4% della variabilità. La componente è quasi indipendente dalla dimensione della lesione.

- R² = 0.50: il volume spiega la metà. La componente misura soprattutto quanto è grande la lesione.

Si calcola sia sulle componenti ruotate sia su quelle non ruotate, per vedere cosa cambia con la rotazione. Il volume è la colonna scelta all'inizio, uguale a quella dei metadati dei soggetti.

> Il paper calcola l'R² sulle componenti ruotate e riporta R² < 0.1 per tutte: le legge come relativamente indipendenti dalla dimensione della lesione.
>
> Nel paper (Risultati): *"All components were relatively independent from the lesion size (all R² < 0.1; see Supplementary Data 1 for a full list of the 46 components pattern of disconnection)."*

In [ ]:
registry = load_participants_registry().set_index("subject_id")
missing = sorted(set(metadata["subject_id"]) - set(registry.index))
if missing:
    raise ValueError(f"{len(missing)} soggetti della matrice non sono nel registro: {missing[:5]}")
volume = registry.loc[metadata["subject_id"], LESION_VOLUME_COLUMN].to_numpy(dtype=float)
if np.isnan(volume).any():
    raise ValueError(f"{int(np.isnan(volume).sum())} soggetti senza {LESION_VOLUME_COLUMN}")

r2_rotated = squared_correlation(scores, volume)
r2_unrotated = squared_correlation(pca.pca_scores(N_COMPONENTS), volume)

print(f"Volume della lesione ({LESION_VOLUME_COLUMN}): mediana {np.median(volume):.0f} voxel, da {volume.min():.0f} a {volume.max():.0f}")
print()
for name, r2 in (("ruotate", r2_rotated), ("non ruotate", r2_unrotated)):
    print(f"Componenti {name}: R² massimo {r2.max():.3f}, mediano {np.median(r2):.3f}, {int((r2 > 0.1).sum())} su {len(r2)} sopra 0.1")

print()
for k in np.argsort(r2_rotated)[::-1][:5]:
    print(f"Componente ruotata {k + 1}: R² con il volume = {r2_rotated[k]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
positions = np.arange(1, N_COMPONENTS + 1)
ax.bar(positions - 0.2, r2_rotated, width=0.4, label="ruotate (varimax)", color="tab:blue")
ax.bar(positions + 0.2, r2_unrotated, width=0.4, label="non ruotate (PCA)", color="tab:orange")
ax.axhline(0.1, color="tab:red", linestyle="--", linewidth=1, label="soglia 0.1 del paper")
ax.set_xlabel("Componente")
ax.set_ylabel("R² con il volume della lesione")
ax.set_title(f"{SESSION_DIR.name}: dipendenza dal volume della lesione")
ax.legend()
plt.show()

## 6. Mappe delle componenti nel cervello

**Cosa si vuole vedere.** Le componenti ruotate dicono quali voxel pesano insieme. La mappa risponde a una domanda diversa: in quali voxel la disconnessione dei soggetti cambia davvero quando cambia il punteggio di una componente? Così si vede dove, nel cervello, si trova ogni componente.

**Passo 1: una regressione per voxel.** Per ogni voxel si prende il suo valore di disconnessione nei 5845 soggetti e lo si mette in relazione con i 42 punteggi dei soggetti, tutti insieme (regressione multipla). Per ogni componente si ottiene un coefficiente: di quanto cambia la disconnessione del voxel quando il punteggio di quella componente sale di 1, con le altre componenti tenute ferme.

**Passo 2: la statistica *t*.** Il coefficiente da solo non dice se è affidabile. Lo si divide per l'incertezza con cui è stimato (l'errore standard) e si ottiene *t*.

- *t* = 10: l'effetto è dieci volte più grande della sua incertezza. È un effetto solido.

- *t* vicino a 0: l'effetto non si distingue da zero.

- Il segno dà la direzione: positivo, la disconnessione del voxel cresce con il punteggio; negativo, cala.

Il risultato è, per ogni componente, un valore *t* per ogni voxel. Questa è la **mappa** della componente.

> Nel paper (Metodi, «Components maps»): *"We used a permuted (n = 1000) multiple regression to statistically assess the relationship between each component and voxel in the white matter. To do so, we used the component score as an independent variable and the voxel probability of disconnection in the stroke disconnectome maps as dependent variables."*
>
> Il paper usa `randomise` di FSL con 1000 permutazioni; qui le mappe sono statistiche *t* senza test di permutazione. Le mappe del paper sono nella sua Fig. 3.

**Soglia di visualizzazione.** Con 5845 soggetti anche effetti piccoli hanno *t* alto (qui il massimo va da circa 70 a 420 a seconda della componente). Una soglia uguale per tutte, ad esempio |*t*| > 3, colorerebbe quasi tutto il cervello. Si colorano invece i voxel con |*t*| di almeno una frazione del massimo di quella componente.

**Segno della componente.** Il segno di una componente è arbitrario: invertire i suoi pesi e i suoi punteggi dà lo stesso risultato. Ogni mappa è orientata in modo che il voxel con |*t*| più grande sia positivo, così le mappe si confrontano senza dipendere da questa scelta.

**Riproducibilità.** La domanda è: la mappa dipende dai soggetti scelti? Si fa così:

1. si dividono a caso i soggetti in due metà;

2. si rifà la regressione in ciascuna metà, ottenendo due mappe per ogni componente;

3. si calcola la correlazione (Pearson, sui voxel) tra le due mappe.

La correlazione vale 1 se le due mappe sono uguali e 0 se non hanno niente in comune. Un valore alto vuol dire che la componente si ritrova allo stesso modo in due gruppi diversi di soggetti.

Le due metà non rifanno la PCA: usano i punteggi calcolati su tutti i soggetti (sezione 4). Mostrano quindi se la stessa componente si ritrova, non se la PCA stessa è stabile.

> Nel paper (Metodi, «Components maps»): *"the 1333 disconnectome maps dataset was split in two 666 datasets and the multiple regression was run twice. Quality of the results duplication was assessed by means of Pearson correlations between the two set of maps derived from the analyses."* E nei Risultati: *"Pearson correlation indicated a good reproducibility across the two sets of component maps (average Pearson R = 0.813 ± 0.079)."*
>
> Il paper riporta 0.813 ± 0.079 in media, con componenti da 0.39 a 0.98. I valori non sono direttamente confrontabili con i nostri: qui le metà sono da circa 2900 soggetti invece di 666, la correlazione è su tutti i voxel e i punteggi sono comuni alle due metà.

I valori da modificare:

- `COMPONENTS_TO_PLOT`: i numeri delle componenti da disegnare, da 1 a 42. Esempi: `[1, 2, 3, 4]` per quattro, `list(range(1, 9))` per le prime otto, `list(range(1, 43))` per tutte.

- `MAP_THRESHOLD_FRACTION`: la quota del picco di ciascuna componente sotto cui un voxel non viene colorato. Con `0.3` si vedono i voxel con |*t*| di almeno il 30% del massimo di quella componente. Più è alta, più la mappa si restringe al nucleo.

- `SPLIT_SEED`: il seme del sorteggio delle due metà. Cambiandolo si cambia la divisione.

In [ ]:
COMPONENTS_TO_PLOT = [1, 2, 3, 4]
MAP_THRESHOLD_FRACTION = 0.3
SPLIT_SEED = 0

if not 0.0 <= MAP_THRESHOLD_FRACTION < 1.0:
    raise ValueError(f"MAP_THRESHOLD_FRACTION deve stare tra 0 e 1 (1 escluso), trovato {MAP_THRESHOLD_FRACTION}")
bad = [k for k in COMPONENTS_TO_PLOT if not 1 <= k <= N_COMPONENTS]
if bad or len(set(COMPONENTS_TO_PLOT)) != len(COMPONENTS_TO_PLOT):
    raise ValueError(f"COMPONENTS_TO_PLOT deve avere numeri distinti tra 1 e {N_COMPONENTS}, trovato {COMPONENTS_TO_PLOT}")

# Le due metà: sorteggio dei soggetti con un seme fisso, così il risultato si ripete.
order = np.random.default_rng(SPLIT_SEED).permutation(X.shape[0])
half_a = np.sort(order[: X.shape[0] // 2])
half_b = np.sort(order[X.shape[0] // 2 :])
print(f"Metà A: {len(half_a)} soggetti, metà B: {len(half_b)} soggetti (seme {SPLIT_SEED})")

MAPS_CACHE = CACHE_DIR / "maps.npz"
MAPS_FINGERPRINT = CACHE_DIR / "maps_source.json"
maps_source = {**varimax_source, "split_seed": SPLIT_SEED}

if USE_CACHE and MAPS_CACHE.is_file():
    if json.loads(MAPS_FINGERPRINT.read_text()) != maps_source:
        raise ValueError(f"la cache delle mappe in {CACHE_DIR} ha altri parametri: metti USE_CACHE = False per rifarla")
    cached = np.load(MAPS_CACHE)
    t_full, t_a, t_b = cached["t_full"], cached["t_a"], cached["t_b"]
    n_degenerate = cached["n_degenerate"]
    print(f"Mappe lette dalla cache: {MAPS_CACHE}")
else:
    maps_full, maps_a, maps_b = component_t_maps(X, scores, [np.arange(X.shape[0]), half_a, half_b])
    t_full, t_a, t_b = (m.t.astype(np.float32) for m in (maps_full, maps_a, maps_b))
    n_degenerate = np.array([m.n_degenerate_voxels for m in (maps_full, maps_a, maps_b)])
    np.savez(MAPS_CACHE, t_full=t_full, t_a=t_a, t_b=t_b, n_degenerate=n_degenerate)
    MAPS_FINGERPRINT.write_text(json.dumps(maps_source))
    print(f"Mappe calcolate e salvate in {MAPS_CACHE}")

print(f"Voxel senza varianza residua (t messo a 0): {n_degenerate[0]} su tutti i soggetti, {n_degenerate[1]} nella metà A, {n_degenerate[2]} nella metà B")

# Stessi segni per le due metà e per la mappa completa: la correlazione tra metà non dipende dal segno scelto.
signs = positive_peak_signs(t_full)
t_full, t_a, t_b = (t * signs[:, None] for t in (t_full, t_a, t_b))
reproducibility = map_reproducibility(t_a, t_b)

In [ ]:
print(f"Riproducibilità tra le due metà, media {reproducibility.mean():.3f} ± {reproducibility.std(ddof=1):.3f} (paper: 0.813 ± 0.079)")
print(f"Da {reproducibility.min():.3f} (componente {reproducibility.argmin() + 1}) a {reproducibility.max():.3f} (componente {reproducibility.argmax() + 1})")
print()
for k in range(N_COMPONENTS):
    print(f"Componente {k + 1}: riproducibilità = {reproducibility[k]:.3f}, t massimo in valore assoluto = {np.abs(t_full[k]).max():.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(1, N_COMPONENTS + 1), reproducibility, color="tab:blue")
ax.axhline(0.813, color="tab:red", linestyle="--", linewidth=1, label="media del paper (0.813)")
ax.set_xlabel("Componente")
ax.set_ylabel("Riproducibilità tra le due metà")
ax.set_title(f"{SESSION_DIR.name}: riproducibilità delle mappe")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()

### Le mappe

Una figura per componente, con tre sezioni del cervello (sagittale, coronale, assiale) che passano per il voxel con *t* più grande. In rosso gli effetti positivi, in blu quelli negativi: dopo l'orientamento il picco è positivo, ma la componente può pesare anche con segno opposto altrove. Lo sfondo è il template MNI T1 a 1 mm.

Le sezioni sono sul template T1, senza le etichette anatomiche (sigle dei fasci) né funzionali.

> Nel paper (Risultati, Fig. 3) le mappe sono sulla sostanza bianca: *"we estimated the white matter structure subjacent to each component allowing to derive a statistical map of the function of white matter connections."*

In [ ]:
mask = np.load(SESSION_DIR / "non_constant_mask.npy")
# Geometria delle mappe: la griglia a 2 mm (solo intestazione). Sfondo: il T1 a 1 mm, perché il file T1 a 2 mm ha i dati troncati.
template = nib.load("assets/templates/tpl-MNI152NLin6Asym_res-2_T1w.nii.gz")
background = nib.load("assets/templates/tpl-MNI152NLin6Asym_res-1_T1w.nii.gz")
if mask.size != int(np.prod(template.shape)) or int(mask.sum()) != X.shape[1]:
    raise ValueError(f"la maschera ({mask.size} voxel, {int(mask.sum())} veri) non corrisponde al template {template.shape} o alla matrice ({X.shape[1]} colonne)")


def map_to_image(values):
    # Le colonne di X sono i voxel della maschera, nell'ordine in cui la griglia è stata srotolata (C-order).
    volume = np.zeros(mask.size, dtype=np.float32)
    volume[mask] = values
    return nib.Nifti1Image(volume.reshape(template.shape), template.affine)


for k in COMPONENTS_TO_PLOT:
    t_map = t_full[k - 1]
    peak_column = int(np.argmax(np.abs(t_map)))
    peak_voxel = np.unravel_index(np.flatnonzero(mask)[peak_column], template.shape)
    peak_mm = nib.affines.apply_affine(template.affine, peak_voxel)
    plotting.plot_stat_map(
        map_to_image(t_map),
        bg_img=background,
        threshold=MAP_THRESHOLD_FRACTION * abs(t_map[peak_column]),
        cut_coords=tuple(peak_mm),
        display_mode="ortho",
        title=f"Componente {k}: t massimo {t_map[peak_column]:.0f}, riproducibilità {reproducibility[k - 1]:.2f}",
    )
    plt.show()